# Airspeed velocity demo

First, we need to specify the `asv` config. In particular, it is important to choose the `existing` environment type - currently, it is the only way to stay with `uv`.

In [ ]:
%%writefile asv.conf.json
{
  "version": 1,
  "project": "asv_ysda_demo",
  "repo": ".",
  "environment_type": "existing",
  "benchmark_dir": "benchmarks",
  "build_command": [],
  "matrix": {},
  "results_dir": "results"
}


Imagine we're doing some ML (no way) and implemented `nn.SiLU()` with our bare hands. After all, everything happens!

In [ ]:
%%writefile mypackage/silu.py
import torch
import typing as tp

def get_silu() -> tp.Callable[[torch.Tensor], torch.Tensor]:
    def fn(x):
        return x * x.sigmoid()
    return fn

Now we're going to code a **benchmark** for it. We will need to define the methods which names starts from `time_` and `peakmem_`. The `setup` function is required for running code that is neccesary for the tests but should not be measured by them.

In [ ]:
%%writefile benchmarks/bench_ops.py
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))


import torch
from mypackage.silu import get_silu

def check_cuda_avail():
    if not torch.cuda.is_available():
        raise NotImplementedError(
            "Torch could not detect your GPU, so..."
        )


class TimeFusionCUDA:
    params = [1024, 4096]
    param_names = ["size"]

    def setup(self, size):
        check_cuda_avail()

        self.x = torch.randn(size, size, device="cuda")
        # compile if necessary
        self.silu = get_silu()
        # warmup (avoid measuring compile)
        self.silu(self.x)
        torch.cuda.synchronize()

    def time_silu(self, size):
        self.silu(self.x)
        torch.cuda.synchronize()


class MemFusionCUDA:
    params = [1024, 4096]
    param_names = ["size"]

    def setup(self, size):
        check_cuda_avail()
        
        self.x = torch.randn(size, size, device="cuda")
        self.silu = get_silu()
        self.silu(self.x)
        torch.cuda.synchronize()

    def peakmem_silu(self, size):
        self.silu(self.x)
        torch.cuda.synchronize()


Okay, let's commit the changes - we worked really hard today.

In [ ]:
%%bash
git commit -am 'Silly silu'

And here we're going to run and remember the benchmark.

In [ ]:
%%bash
uv run asv run --python=same --set-commit-hash=$(git rev-parse HEAD)

A few days after you and your bro are shocked by existence of `torch.compile` (thx effdl'26)! Of course, the next step is to apply it to your latest job result and to show off.

In [ ]:
%%writefile mypackage/silu.py
import torch
import typing as tp

def get_silu() -> tp.Callable[[torch.Tensor], torch.Tensor]:
    def fn(x):
        return x * x.sigmoid()

    return torch.compile(fn)

New commit:

In [ ]:
%%bash
git commit -am 'Compiled silu'

Another bench:

In [ ]:
%%bash
uv run asv run --python=same --set-commit-hash=$(git rev-parse HEAD)

And you're finally deserved that web ui link to which you probably did not even noticed in the `README.md`:

In [ ]:
%%bash

uv run asv publish
uv run asv preview